# 16장 · 청킹 · 문자열 경계와 겹침 길이 정하기

청킹은 긴 글을 검색할 수 있는 작은 조각으로 나누는 작업입니다. 겹침은 앞 조각 끝의 일부를 다음 조각의 시작에도 남기는 방법입니다.

## 슬라이스의 시작 위치로 겹침 정하기

`text[:4]`는 처음 네 글자를, `text[2:]`는 세 번째 글자부터 끝까지를 꺼냅니다. 두 조각 모두에 어떤 글자가 남는지 출력으로 확인하세요.

In [ ]:
text = "가나다라마바"
print(text[:4])
print(text[2:])

가나다라
다라마바


앞 조각은 위치 0부터 3까지, 뒤 조각은 2부터 끝까지 읽었습니다. “다라”가 두 조각에 남습니다. 두 번째 시작 위치를 4로 바꾸면 겹치는 글자가 없어집니다.

## 응용 실습 · PDF의 조건과 예외 나누기

## 실습 준비

전체 실습 묶음을 풀고 노트북을 엽니다. 같은 폴더에서 시작해 자료 묶음의 `pre-mid/documents`를 읽습니다. 공용 준비 파일에 뒤에서 사용할 함수도 있으므로 **아래 셀에서 가져오는 이름**부터 살펴보세요. pypdf와 python-dotenv를 사용합니다. 모델 호출은 없습니다.

In [1]:
from pathlib import Path
import sys

# 압축을 푼 폴더 또는 그 아래에서 실습 자료의 위치를 찾습니다.
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pre-mid/documents/rag/rag_lab.py").is_file()), None)
if root is None:
    raise FileNotFoundError("전체 실습 묶음을 풀고 pre-mid 폴더 옆에서 실행하세요.")
rag_dir = root / "pre-mid/documents/rag"
sys.path.insert(0, str(rag_dir))
from portfolio import load_step, save_step, page_id


### 내 페이지 목록에서 청크 만들기

15장에서 모은 여섯 쪽을 엽니다. `page_chunks(pages, size)`는 각 쪽 안에서 글자를 나누고 파일명과 쪽수를 조각마다 복사합니다. `size=None`이면 쪽을 통째로 보존합니다. 이 기준선으로 다음 장의 검색을 먼저 시험하세요.

In [ ]:
from rag_lab import page_chunks
source_pages = load_step(root, "pages")
size = None
my_chunks = page_chunks(source_pages, size=size)
save_step(root, "chunks", my_chunks)
print(len(source_pages), len(my_chunks), my_chunks[0]["source"], my_chunks[0]["page"])

6 6 recorder-a-1.0.pdf 1


기본 자료에서 첫 두 수는 모두 6입니다. 17장은 `my-course/rag/chunks.json`을 직접 읽습니다. `size`를 95로 바꿔 다시 저장하면 검색기에 들어갈 조각도 달라집니다. 같은 질문을 비교하려면 처음 사용한 크기를 적어 두세요.

### 실습 · 조각 수와 쪽수 구분하기

원본을 95자씩 나눈 뒤 조각 수가 늘었는지 확인하세요. 서로 다른 출처 쪽의 수는 처음과 같아야 합니다.

In [ ]:
trial = page_chunks(source_pages, size=95)
# (파일명, 쪽수) 쌍을 세어 원본 수와 비교하세요.

<details><summary>정답 코드·출력·풀이</summary><pre><code>print(len(trial) &gt; len(source_pages), len({(r[&quot;source&quot;], r[&quot;page&quot;]) for r in trial}) == len(source_pages))</code></pre><p>제공 자료로 확인할 값</p><pre>True True</pre><p>한 쪽에서 나온 여러 조각은 서로 다른 조각 ID를 가져도 출처 쪽은 같습니다.</p></details>

## str.index로 분할할 문장의 위치 찾기

`text.index("이미 승인")`은 그 문구가 처음 시작하는 위치를 정수로 돌려줍니다. `text.index("장비 종류", start_at)`은 앞서 찾은 위치부터 다시 찾습니다. 두 위치를 슬라이스의 양 끝에 넣으면 그 사이의 글만 남습니다. 문구가 없으면 `ValueError`가 납니다.

loan-summer.pdf 2쪽을 먼저 읽고 “수령 시간만 바꾸려는데 처음부터 신청해야 하나요?”에 밑줄을 그어 답하세요. “다시 적용하지 않습니다”라는 문장만 보면 **무엇을** 다시 적용하지 않는지 모릅니다. 답이 되는 최소 범위에는 대상과 부정 표현이 함께 있어야 합니다.

`text.index("이미 승인")`은 해당 문구가 시작되는 글자의 위치를 돌려줍니다. `text.index("장비 종류", start_at)`의 두 번째 인자는 찾기 시작할 위치입니다. 슬라이스 `text[시작:끝]`은 끝 위치의 글자를 포함하지 않습니다. PDF 추출 결과에서 띄어쓰기가 달라지면 `index`에서 ValueError가 나므로 원문 한 쪽부터 출력해 확인하세요.

In [2]:
from pypdf import PdfReader

text = PdfReader(rag_dir / "loan-summer.pdf").pages[1].extract_text().strip()
start_at = text.index("이미 승인")
passage = text[start_at:text.index("장비 종류", start_at)].strip()
boundary = passage.index("위 신규")
# 조건 뒤에서 자르면 뒷조각의 '위'가 가리키는 대상을 잃습니다.
print(passage[:boundary])
print("-----")
print(passage[boundary:])

이미 승인받은 장비의 수령 시간만 바꾸는 경우에는

-----
위 신규 신청 마감을 다시 적용하지 않습니다.
수령 전날 오후 4시까지 담당자에게 새 시간을 알려 주세요.


## 문자열 슬라이스로 겹침 범위 조절하기

뒤 조각의 시작에 앞 조각의 일부를 남기면 대상을 되찾을 수 있습니다. 그러나 겹침이 너무 짧으면 중요한 명사가 여전히 빠지고, 너무 길면 같은 문장이 상위 후보 여러 칸을 차지할 수 있습니다. 정답에 필요한 문구가 실제로 남았는지 직접 보세요.

In [3]:
overlap = 26
print(passage[max(0, boundary - overlap):])
# 두 조각 모두 같은 PDF의 2쪽에서 왔습니다.
print("loan-summer.pdf", 2)

 승인받은 장비의 수령 시간만 바꾸는 경우에는
위 신규 신청 마감을 다시 적용하지 않습니다.
수령 전날 오후 4시까지 담당자에게 새 시간을 알려 주세요.
loan-summer.pdf 2


## 겹침 길이와 출처 쪽수 함께 보존하기

A형 설명서처럼 한 쪽이 짧고 같은 주제를 다루면 페이지 한 장을 기준선으로 삼을 수 있습니다. 긴 운영 편람에서는 한 쪽에 서로 다른 규칙이 섞일 수 있습니다. 문단 제목·판본·쪽을 그대로 가진 조각을 만들고, 조건과 결론이 분리된 곳을 조사합니다.

위 예제의 `overlap`은 두 번째 조각을 경계보다 앞에서 시작시키는 글자 수입니다. 아래 교재 함수 `page_chunks(pages, size=95)`에는 겹침 인자가 없습니다. 각 쪽을 처음부터 95글자씩 잘라 새 ID를 붙입니다. `size`를 생략하면 한 쪽을 통째로 보관합니다. 크기를 줄였을 때 출처 쪽수와 조각 번호 중 무엇이 바뀌는지 확인하세요.

### 겹침 두 글자가 남기는 것

In [ ]:
sentence = "신청 마감은 전날"
cut = sentence.index("전날")
print(sentence[cut:])
print(sentence[cut - 2:])

전날
은 전날


두 번째 출력에는 경계 바로 앞 두 글자까지 포함됩니다. 겹침을 늘렸다고 조건 전체가 자동으로 들어오지는 않습니다. “신청 마감”까지 읽으려면 시작 위치를 어디로 옮겨야 할까요?

In [4]:
from rag_lab import SOURCES, read_pages, page_chunks
pages = read_pages(SOURCES / "recorder-a-1.1.pdf")
whole = page_chunks(pages)
short = page_chunks(pages, size=95)
print("쪽 단위", len(whole), "짧은 조각", len(short))
for row in short[:2]:
    print(row["id"], row["text"])

쪽 단위 2 짧은 조각 7
recorder-a-1.1.pdf#p1-c1 들소리 A형   /   휴대용 녹음기
녹음 표시등
1.1판    적용 기간   2027년 4월 1일부터
표시등 설정
녹음 중에는 앞면의 빨간 표시등이 켜집니다. 녹음하는 
recorder-a-1.1.pdf#p1-c2 동안
표시등을 끄는 설정은 없습니다.
화면과 표시등
화면 밝기를 낮춰도 표시등은 켜져 있습니다. 조용한 방에서
화면을 어둡게 하고 싶을 때는 설정에서 화면 밝기를 낮추세요


## 같은 질문을 두 분할에서 찾아보기

질문은 “이미 승인받은 장비의 **수령 시간만** 바꾸려면 언제까지 알려야 하나요?”입니다. 여기서는 PDF 찾기 창처럼 정확히 같은 문구가 든 조각을 꺼냅니다. 그래야 검색 방법을 바꾸지 않고 분할 경계만 비교할 수 있습니다. 첫 분할은 대상 뒤에서 잘랐고, 두 번째 분할은 대상과 마감을 함께 보존했습니다.

In [5]:
question_words = "수령 시간만"
bad = [passage[:boundary], passage[boundary:]]
good = [passage]
for label, chunks in [("조건 뒤에서 자름", bad), ("예외 한 단락 보존", good)]:
    found = next(chunk for chunk in chunks if question_words in chunk)
    print(label, "→", found)
    print("받은 조각에 마감이 있는가?", "오후 4시" in found)

조건 뒤에서 자름 → 이미 승인받은 장비의 수령 시간만 바꾸는 경우에는

받은 조각에 마감이 있는가? False
예외 한 단락 보존 → 이미 승인받은 장비의 수령 시간만 바꾸는 경우에는
위 신규 신청 마감을 다시 적용하지 않습니다.
수령 전날 오후 4시까지 담당자에게 새 시간을 알려 주세요.
받은 조각에 마감이 있는가? True


첫 조각은 문의자가 말한 표현을 정확히 담아 찾아냈지만 마감이 빠졌습니다. 두 번째는 같은 문구를 찾아도 전날 오후 4시까지라는 답을 줄 수 있습니다. 여기서 효과를 낸 변경은 검색 알고리즘이 아니라 조각의 경계입니다. 반대로 긴 조각을 쓰면 ‘시간만 변경’과 ‘장비 종류 변경’이 한꺼번에 들어와 어느 조건이 해당하는지 다시 읽어야 합니다.

조건과 결론이 아예 서로 다른 쪽에 걸쳐 있으면 쪽 경계를 넘겨 붙일 때 양쪽 출처를 보존해야 합니다. 예를 들어 첫 쪽 끝에 “다음 항목에 해당하면 예외”가 있고 다음 쪽 첫 줄에 그 항목이 있으면, 한 쪽만 인용한 답은 검토하기 어렵습니다. 수업 자료처럼 예외 문단이 같은 쪽에 있는지 먼저 확인하세요.

### 실습 · 길이를 바꿔도 출처가 남을까요?

수업 함수 page_chunks로 다섯 글자를 두 글자씩 나누세요. 각 조각의 글, ID, 원래 쪽수를 출력해 마지막 조각이 한 글자일 때도 출처를 찾을 수 있는지 확인하세요.

In [ ]:
rows = [{"id": "p2", "page": 2, "text": "가나다라마"}]
# page_chunks를 불러와 조각의 세 값을 출력하세요.

<details><summary>정답 코드와 실행 결과</summary><pre><code>from rag_lab import page_chunks
rows = [{&quot;id&quot;: &quot;p2&quot;, &quot;page&quot;: 2, &quot;text&quot;: &quot;가나다라마&quot;}]
for piece in page_chunks(rows, size=2):
    print(piece[&quot;text&quot;], piece[&quot;id&quot;], piece[&quot;page&quot;])</code></pre><pre>가나 p2-c1 2
다라 p2-c2 2
마 p2-c3 2</pre><p>각 ID의 c 뒤 번호는 조각 순서입니다. page는 원래 쪽수이므로 세 조각 모두 2입니다. 이 함수는 일정 길이로 자르며 조각 사이에 겹침을 두지 않습니다.</p></details>

## 실습 · 한 조각으로 답할 수 있는지 검토하기

A형 1.1판에서 “파일이 바뀌면 녹음이 멈추나요?”를 질문으로 씁니다. 60글자와 180글자로 각각 나누고 45분·계속 녹음이 같은 조각에 있는지 찾으세요. 값이 함께 남아도 어느 판본인지 다시 열 수 있는지 확인합니다. 뒷조각에 말의 대상이 빠졌다면 어떤 경계를 옮길지 적으세요.

In [6]:
sizes_to_try = [60, 180]
needle = "45분"
# 각 크기로 page_chunks(pages, size=크기)를 호출하세요.
# needle을 담은 조각의 id와 text를 출력하고 녹음의 지속 여부까지 적혀 있는지 읽으세요.

<details class="notebook-answer"><summary>모범 풀이와 판정 기준</summary><pre><code>for size in [60, 180]:
    chunks = page_chunks(pages, size=size)
    for row in chunks:
        if &quot;45분&quot; in row[&quot;text&quot;]:
            print(size, row[&quot;id&quot;], row[&quot;text&quot;])</code></pre><p>‘45분’을 찾았더라도 녹음의 지속 여부가 다른 조각에 있으면 그 조각 하나로는 부족합니다. 두 설정 모두 우연히 충분했다면 더 작은 크기로 바꿔 실패 경계를 찾아도 됩니다.</p><p>집필 시 실행 결과</p><pre>60 recorder-a-1.1.pdf#p2-c2 
연속 녹음
한 번에 길게 녹음하면 45분마다 새 파일이 열립니다. 파일이
바뀌어도 녹음은 계속됩니다.
파일
180 recorder-a-1.1.pdf#p2-c1 들소리 A형   /   휴대용 녹음기
파일 저장과 복사
1.1판    적용 기간   2027년 4월 1일부터
연속 녹음
한 번에 길게 녹음하면 45분마다 새 파일이 열립니다. 파일이
바뀌어도 녹음은 계속됩니다.
파일 옮기기
녹음을 끝낸 뒤 케이블로 컴퓨터에 연결하세요. 녹음기에 있는
RECORD 폴더를 열고 필요한 파일을</pre></details>

## 독립 실습 · 어느 분할이 실제 문의에 유리한가

대여 안내 개정판 2쪽을 두 장의 메모로 나눠 동료에게 건네세요. 첫 시도는 “위 신규 신청 마감” 앞에서, 둘째 시도는 “장비 종류까지” 앞에서 나눕니다. 동료는 한 조각만 읽고 “수령 시간만 바꿀 때 언제까지 연락하는가?”에 답합니다. 원문을 다시 붙이지 않고도 이해할 수 있게 제목 또는 앞 문장을 얼만큼 가져올지 합의하세요.

| 분할 위치 | 학생이 받은 조각 | 빠진 조건 | 선택할 경계와 이유 |
|---|---|---|---|
| 첫 시도 | | | |
| 둘째 시도 | | | |

<details class="notebook-answer"><summary>판정 예시와 해설</summary><p>“위 신규 신청 마감을 다시 적용하지 않습니다”만 남으면 어느 사용자의 요청인지 알 수 없습니다. 앞의 “이미 승인받은 장비의 수령 시간만 바꾸는 경우”와 뒤의 “수령 전날 오후 4시까지”를 같은 조각에서 확인하는 것이 좋습니다. 장비 **종류** 변경은 신규 신청이라는 문장을 다음 조각으로 분리할 수 있지만, 그 조각에서도 무엇에 대한 신규 신청인지 제목과 출처가 남아야 합니다.</p></details>

## 다음 수업에 가져갈 것

17장이 실제로 읽는 파일은 `my-course/rag/chunks.json`입니다. 처음 검색할 때는 한 쪽을 한 청크로 저장하고, 두 번째 실험에서는 크기만 바꿔 같은 질문을 검색하세요.

우리 기준 색인은 **페이지 하나를 청크 하나**로 사용합니다. 다른 크기에서 개선된 질문이 있으면 그 질문과 조각 ID를 남기세요. 다음 장의 검색 순위와 비교할 관찰 기록입니다.